# 04 · Evaluación, latencia y coste

Se mide lo que el producto promete: que los campos del PDF coinciden con la verdad terreno, que la tesis no publica cifras huérfanas, y cuánto cuesta la pasada frente a un escenario de API.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from helios.analysis.cost import estimate_cost
from helios.analysis.score import score_fields
from helios.analysis.textutil import keyword_hit_rate, wer
from helios.config import Settings
from helios.demo.case import load_demo_case, load_script, load_truth
from helios.ingest.pdf import read_pdf
from helios.orchestration.pipeline import run_analysis

case = load_demo_case()
truth = load_truth()
document = read_pdf(case.pdf_path)
fields = score_fields(document.fields, truth["fields"])
print("F1 campos PDF", round(fields["f1"], 3))
print("faltan", fields["missing"], "distintos", fields["mismatched"])


In [ ]:
result = run_analysis(case, use_models=True)
print("respaldo del borrador", f"{result.raw_support_ratio:.0%}")
print("respaldo publicado", f"{result.published_support_ratio:.0%}")
print("modo", result.thesis_mode)
for stage in result.stages:
    print(f"{stage.name:28} {stage.seconds:7.2f}s  {stage.status:8} caché={stage.cached}")
cost = estimate_cost(result.stages, Settings.from_env(), briefing_chars=len(result.briefing_script))
print("kWh", round(cost["kwh"], 6), "EUR local", round(cost["local_eur"], 6), "EUR API hipotética", round(cost["api_eur"], 6))
print(cost["note"])


## Transcripción

La referencia es el guion, no una grabación de un tercero. Si Whisper no está instalado, la celda enseña la métrica con un ejemplo y no finge un error de palabra.

In [ ]:
script = load_script()
print("WER de un texto idéntico", wer(script, script))
try:
    from helios.models.whisper_asr import transcribe
    transcript, cached = transcribe(case.audio_path, Settings.from_env())
    print("WER", round(wer(script, transcript.text), 3), "caché" if cached else "frío")
    print("palabras clave", round(keyword_hit_rate(transcript.text, truth["transcript_keywords"]), 3))
    print(transcript.text)
except Exception as exc:
    print("Whisper no ejecutado:", exc)
    print("cobertura de ejemplo", keyword_hit_rate(script, truth["transcript_keywords"]))


Una pasada en caliente, con caché, no sirve para prometer latencia. El presupuesto de diseño está en el README: visión y transcripción dominan el tiempo en CPU. El coste local de la tabla es electricidad, no una factura de API.